# Getting started with yabadaba

This Notebook provides a step-by-step guide for creating a new data project using yabadaba.  

## 1. Decide what schema to use

The first step is to decide what schema to use for representing your data.  This might sound scary and complicated for non-data scientists, but ends up being rather simple when you start building it.

Basic terminology:
- data: The content you wish to capture and save for later.
- metadata: Data fields that are used to characterize other data.  For scientific data, you can think of the primary data being the results, and metadata describing the who, what, where, and how the data was obtained.
- FAIR principles: These are a set of guiding principles for data and databases to "improve the Findability, Accessibility, Interoperability, and Reuse of digital assets".
- format: Data formats dictate how data is organized when saved to a file. Common choices are JSON, XML, YAML, CSV, or a custom .txt file.
- schema: Data schemas map out what values and value types are collected together to represent a full dataset.
- record: This is used by yabadaba both to refer to a Record object as well as a data entry that adheres to a specific schema.


How to design a schema:
1. Think about what data and metadata should be included in the schema.
    1. The general guideline for metadata is to include as much as possible. Essentially, if it is something somebody may wish to know about your data or search for in the future it should be included. Who created it, what were the conditions or input settings, where can more information (i.e. a related publication) be found, how was the data processed and analyzed, ...
    2. Most database infrastructures have a limit on the size of records that can be used.  As such, primary data should be divided between "raw" and "final" data, with the final processed data present in the record, and raw data either in the record if it is small and simple or stored elsewhere if it is large and/or complex. For externally stored raw data, the record can then point to it by listing a file name, url, etc.
    3. What is the data's type, i.e. str, integer, float/real number, list/array...? If the data is complex, can it easily be represented with one or more simple data fields?
 2. Optionally, investigate if there are existing schemas for the same or similar data.
    1. Reusing schemas, schema components, or element naming conventions can help make your schemas easier to build and interpret.
    2. If there is an existing schema, does it capture all the data and metadata you care about or should it be extended/modified?
    3. Do not worry too much if you cannot find an existing schema you like or just want to get started.  In my personal opinion, a specific format and schema are not that important as long as the schema you use contains all the important metadata.
 3. Think about how the data and metadata fields should be organized in the schema. The tree-like data formats like JSON and XML allow for grouping related data fields into subsets. Using subsets allows for more of an object-oriented representation, where you can for instance collect all details and settings for a given instrument or simulation. Subsets also support reuse of the schema components as a single record can repeat a subset or the same subset can be shared by multiple record schemas.

## 2. Define yabadaba Record classes

Defining Record classes with yabadaba has been designed to be as easy as possible and consists of little more than defining the record's schema.

Record definition components:
1. Import yabadaba and create a new class that inherits from yabadaba.record.Record.
2. Define a "style" property which gives the Record a style name.
3. Define a "modelroot" property which specifies what the root element of a tree-like data format should be called.  This should be related to the style name.
4. Create an _init_values() method that defines the schema using _add_value() calls.
5. Incorporate the new record into yabadaba by adding it to the yabadaba's recordmanager.

Value definitions, a.k.a. _add_value() parameters:
- style (str): indicates the data type (str, bool, float, date, floatarray, intarray, ...). Required.
- name (str): the name of the value attribute in Python. Required.
- defaultvalue (str, optional): specifies the default value to use.
- valuerequired (bool optional): indicates if a value must be set. Default is False.
- allowedvalues (list, optional): a list of allowed/recommended values.
- allowcustomvalue (bool, optional): if False (default), the value is restricted to those in the allowedvalues list.  
- metadatakey (str, optional): key to use for the value in the metadata representation. Uses name if not given.
- modelpath (str, optional): path (relative to modelroot) to find the value in the tree-like representation. Uses name if not given.
- description (str, optional): informative description of the value.
- unit (str, optional): the units to use for float and floatarray values in the metadata and tree-like model representations. If not given, is assumed to be unitless.
- shape (tuple, optiona): a required array shape for intarray and floatarray values. If not given, any shape (i.e. dimensions) is allowed.

In [1]:
import yabadaba
uc = yabadaba.unitconvert

Let's define a FakeData Record class that captures a creator "name" and "affiliation", a measurement "date", settings "mode" and "temperature", and outputs a "pressure".

In [2]:
class FakeData(yabadaba.record.Record):
    """Class for representing fake example data"""

    ########################## Basic metadata fields ##########################

    @property
    def style(self) -> str:
        """str: The record style"""
        return 'fakedata'

    @property
    def modelroot(self) -> str:
        """str: The root element of the content"""
        return 'fakedata'

    ####################### Define Values and attributes #######################

    def _init_values(self):
        """
        Method that defines the value objects for the Record.  This should
        call the super of this method, then use self._add_value to create new Value objects.
        Note that the order values are defined matters
        when build_model is called!!!
        """
        
        self._add_value('str',
                        'creator',
                        valuerequired = True,
                        modelpath = 'creator.name',
                        description = 'Name of the person who created the data')
        
        self._add_value('str',
                        'affiliation',
                        valuerequired = True,
                        modelpath = 'creator.affiliation',
                        description = 'Affiliation of the creator')
        
        self._add_value('date',
                        'date',
                        valuerequired = True,
                        description = 'Date of measurement')

        self._add_value('str',
                        'mode',
                        defaultvalue = 'a',
                        allowedvalues = ['a', 'b', 'c'],
                        description = 'The mode setting for the measurement')
        
        self._add_value('float',
                        'temperature',
                        valuerequired = True,
                        metadatakey = 'T (K)',
                        unit = 'K',
                        description = 'Temperature at which the measurement was taken')

        self._add_value('float',
                        'pressure',
                        metadatakey = 'P (GPa)',
                        unit = 'GPa',
                        description = 'The measured pressure')
                        

# Add the record to yabadaba's recordmanager with name matching its style
yabadaba.recordmanager.loaded_styles['fakedata'] = FakeData

## 3. Record integration into yabadaba

Notice the last line in the above cell, which adds the FakeData class to yabadaba's recordmanager under the 'fakedata' style name.  By making the recordmanager aware of the Record class, it is now integrated into yabadaba and accessible to yabadaba's features, such as
- New record objects can be created by calling yabadaba.load_record() with the style name.
- Database objects can be used to query, save, modify, and delete records of the style.
- Documentation can be automatically generated based on the record's values to describe all values with yabadaba.valuedoc() and all supported query operations with yabadaba.querydoc().


In [3]:
# Valuedoc generates a markdown-style string.
# Setting render=True will automatically render it as HTML for IPython environments like Jupyter
yabadaba.valuedoc('fakedata', render=True)

# fakedata Values

- __creator__ (*str*): Name of the person who created the data
- __affiliation__ (*str*): Affiliation of the creator
- __date__ (*date*): Date of measurement
- __mode__ (*str*): The mode setting for the measurement
- __temperature__ (*float*): Temperature at which the measurement was taken
- __pressure__ (*float*): The measured pressure

**NOTES ON YABADABA INTEGRATION**: Additional methods are available in yabadaba that support more adaptive and dynamic additions of records to the recordmanager when records are collected into Python packages.
- recordmanager.import_style() can be given the record style name and package path to the class definition. The manager will then attempt to import the record and add it to the recordmanager.  If the import fails, then the error message is saved to the recordmanager instead.  This allows for records with required imports beyond the base package.
- 

In [ ]:
# This is equivalent to "record = FakeData()"
record = yabadaba.load_record('fakedata')

